# 🪟 Python Sliding Window — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A sliding window is like a train car moving along a track.
> You add passengers from the front door and remove them from the back door.
> The car never reverses — it only slides forward.
> The trick: you maintain a constraint (max size, no duplicates, k characters)
> and shrink from the left only when the window violates that constraint.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [What Is Sliding Window? The Visual Model](#1) |
| 2 | [Setup / Initialization](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Variable Window — LC 3](#5) |
| 6 | [Pattern 2: Minimum Window Substring — LC 76](#6) |
| 7 | [Pattern 3: Longest Repeating Replacement — LC 424](#7) |
| 8 | [Pattern 4: Fixed Window Maximum — LC 239](#8) |
| 9 | [Pattern 5: Permutation in String — LC 567](#9) |
| 10 | [The Sliding Window Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>

## 1. What Is Sliding Window? The Visual Model

```
VARIABLE WINDOW — expand right, shrink left on violation

  s = "a b c a b c b b"
  idx:  0 1 2 3 4 5 6 7

  L=0 R=0: window=[a]      ok
  L=0 R=1: window=[a,b]    ok
  L=0 R=2: window=[a,b,c]  ok
  L=0 R=3: window=[a,b,c,a] → duplicate 'a' → shrink left
  L=1 R=3: window=[b,c,a]  ok, max_len=3
  ... continues sliding right

FIXED WINDOW — slide a window of exact size k

  nums = [1, 3, -1, -3, 5, 3, 6, 7]   k=3
  window slides:
  [1, 3,-1]  max=3
     [3,-1,-3] max=3
        [-1,-3,5] max=5
           [-3,5,3] max=5
              [5,3,6] max=6
                 [3,6,7] max=7

  KEY: use a monotonic deque to get max in O(1) per window.
  Deque stores indices in decreasing-value order.
  Evict front when it's outside the window.
  Evict back when back value <= new value (can never be max again).

WINDOW NEVER SHRINKS (LC 424 trick):
  When window is invalid, slide the whole window right instead of shrinking.
  This preserves the best window size seen so far without shrinking.
```

<a id='2'></a>

## 2. Setup / Initialization

In [ ]:
from collections import defaultdict, deque

# VARIABLE WINDOW — set/dict to track window contents
s = "abcabc"
left = 0
window_chars = set()          # for uniqueness constraint
max_len = 0

for right in range(len(s)):   # right pointer always moves forward
    while s[right] in window_chars:  # shrink until constraint satisfied
        window_chars.remove(s[left])
        left += 1
    window_chars.add(s[right])
    max_len = max(max_len, right - left + 1)

print(f"max unique window in '{s}': {max_len}")

# FIXED WINDOW — use deque for O(1) max
nums = [1, 3, -1, -3, 5, 3, 6, 7]
k = 3
dq = deque()          # stores indices, front = current max index
result = []

for i, v in enumerate(nums):
    while dq and dq[0] < i - k + 1:   # evict front if outside window
        dq.popleft()
    while dq and nums[dq[-1]] <= v:    # evict back if smaller than new value
        dq.pop()
    dq.append(i)
    if i >= k - 1:                     # window is full
        result.append(nums[dq[0]])     # front of deque = current window max

print(f"sliding max k={k}: {result}")
print("Setup patterns demonstrated.")

<a id='3'></a>

## 3. The Core API — All Operations

```
OPERATION                         COMPLEXITY   WHAT IT DOES
───────────────────────────────────────────────────────────────────
right += 1 (expand)               O(1)         add element to right of window
left += 1  (shrink)               O(1)         remove element from left
window_map[c] += 1                O(1)         track frequency in window
window_map[c] -= 1; del if 0     O(1)         remove from frequency map
dq.append(i)                      O(1)         add index to monotonic deque
dq.popleft()                      O(1)         evict oldest index (outside window)
dq.pop()                          O(1)         evict smaller values from back
window_size = right - left + 1    O(1)         current window size

THINGS YOU DO NOT DO:
❌  Move left pointer forward past right pointer
❌  Use sliding window on non-contiguous subarrays
❌  Shrink window in LC 424 — slide instead (window never gets smaller)
❌  Forget to evict stale front of deque in fixed window max
❌  Use set instead of dict when you need frequency counts
```

In [ ]:
from collections import defaultdict, deque

# Demo: frequency map window
s = "aabbcc"
window = defaultdict(int)
left = 0
print("Frequency map window expansion:")
for right in range(len(s)):
    window[s[right]] += 1                # add new char to window
    print(f"  add {s[right]}: window={dict(window)}, size={right-left+1}")

print()

# Demo: shrink when window violates constraint (more than 2 unique)
s2 = "araaci"
window2 = defaultdict(int)
left2 = 0
max_k = 2
best = 0
print(f"Longest window with at most {max_k} unique chars in '{s2}':")
for right, ch in enumerate(s2):
    window2[ch] += 1
    while len(window2) > max_k:           # too many unique — shrink
        left_ch = s2[left2]
        window2[left_ch] -= 1
        if window2[left_ch] == 0:
            del window2[left_ch]          # clean up zero-count entries
        left2 += 1
    best = max(best, right - left2 + 1)
print(f"  answer: {best}")
print("Core API demo done.")

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                      WHICH PATTERN
──────────────────────────────────────────────────────────────────
"longest substring without X"             variable window + set
"minimum window containing all of T"      variable window + two freq maps
"longest substring with at most k distinct" variable window + dict
"can replace k chars, longest result"     window never shrinks (LC 424)
"max/min in each sliding window of size k" fixed window + monotonic deque
"does s2 contain a permutation of s1"     fixed window + 26-bucket arrays
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Variable Window — LC 3 Longest Substring Without Repeating Characters

---

```
PROBLEM:
  Find length of longest substring without repeating characters.

TRICK:
  HashMap stores last-seen index of each char.
  When a duplicate is found at right, jump left to just past the
  previous occurrence — not just one step.

SLOW MOTION TRACE on s="abcabcbb":
  seen={}  left=0  max_len=0
  r=0 'a': seen={a:0}, max_len=1
  r=1 'b': seen={a:0,b:1}, max_len=2
  r=2 'c': seen={a:0,b:1,c:2}, max_len=3
  r=3 'a': 'a' seen at 0, left=max(0,0+1)=1, seen={a:3,...}, max_len=3
  r=4 'b': 'b' seen at 1, left=max(1,1+1)=2, seen={b:4,...}, max_len=3
  r=5 'c': 'c' seen at 2, left=max(2,2+1)=3, seen={c:5,...}, max_len=3
  r=6 'b': 'b' seen at 4, left=max(3,4+1)=5, max_len=3
  r=7 'b': 'b' seen at 6, left=max(5,6+1)=7, max_len=3
  answer=3

KEY INSIGHT:
  Store index not just presence — jumping left past the duplicate is O(1)
  instead of sliding one character at a time.

TIME:  O(n) — right pointer visits each char once
SPACE: O(min(n, charset)) — HashMap bounded by charset size
```

In [ ]:
def length_of_longest_substring(s: str) -> int:
    """
    LC 3 — Longest Substring Without Repeating Characters
    Approach: variable window, jump left past last duplicate.
    Args:
        s (str): input string.
    Returns:
        int: length of longest substring without repeated chars.
    Time:  O(n) — each char visited once by right pointer
    Space: O(min(n, charset)) — HashMap at most charset distinct chars
    """
    last_seen = {}    # char → most recent index seen
    left = 0
    max_len = 0

    for right, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= left:
            # jump left past the previous occurrence — O(1) instead of slide
            left = last_seen[ch] + 1
        last_seen[ch] = right          # update to current position
        max_len = max(max_len, right - left + 1)

    return max_len

# Slow motion on "abcabcbb":
# r=3 'a': seen at 0, left jumps to 1
# r=4 'b': seen at 1, left jumps to 2
# max_len stays at 3 throughout

def test_harness(fn):
    tests = [
        ("abcabcbb", 3),
        ("bbbbb",    1),
        ("pwwkew",   3),
        ("",         0),
        ("a",        1),
        ("dvdf",     3),
        ("abcdef",   6),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(length_of_longest_substring)
print("length_of_longest_substring defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Minimum Window Substring — LC 76

---

```
PROBLEM:
  Find minimum window in s containing all chars of t.

TRICK:
  Two frequency maps: need (from t) and window (current).
  Track `matched` = count of chars where window[c] >= need[c].
  When matched == len(need), window is valid. Shrink from left to minimize.

SLOW MOTION TRACE on s="ADOBECODEBANC" t="ABC":
  need={A:1,B:1,C:1} matched_target=3

  Expand until matched=3: window covers ADOBEC (idx 0..5)
    matched=3, window len=6
  Shrink left: remove A(idx 0) → matched=2, stop
  Expand right: add O,D,E,B,A until A found again (idx 10)
    matched=3, window covers DOBEBANC(3..10), but need A,B,C
    shrink: remove D→O→B... until BANC (idx 9..12)
  answer="BANC"

KEY INSIGHT:
  `matched` counter avoids re-checking all chars each step.
  Increment matched only when window[c] exactly reaches need[c].
  Decrement matched only when window[c] drops below need[c].

TIME:  O(|s| + |t|) — each char visited twice (expand + shrink)
SPACE: O(|t|) — two frequency maps bounded by |t|
```

In [ ]:
from collections import Counter

def min_window(s: str, t: str) -> str:
    """
    LC 76 — Minimum Window Substring
    Approach: expand right + shrink left, track matched count.
    Args:
        s (str): source string.
        t (str): target string — find all chars of t in window.
    Returns:
        str: shortest window containing all chars of t, or "" if none.
    Time:  O(|s| + |t|) — expand and shrink each char at most once
    Space: O(|t|) — two frequency maps
    """
    if not t or not s:
        return ""

    need = Counter(t)           # how many of each char we need
    window = {}                 # how many of each char we have
    matched = 0                 # chars where window[c] >= need[c]
    required = len(need)        # distinct chars we need to satisfy

    left = 0
    best = (float('inf'), 0, 0) # (length, left, right)

    for right, ch in enumerate(s):
        window[ch] = window.get(ch, 0) + 1
        if ch in need and window[ch] == need[ch]:
            matched += 1        # just satisfied this char's requirement

        while matched == required:  # valid window — try to shrink
            if right - left + 1 < best[0]:
                best = (right - left + 1, left, right)

            left_ch = s[left]
            window[left_ch] -= 1
            if left_ch in need and window[left_ch] < need[left_ch]:
                matched -= 1    # no longer satisfying this char
            left += 1

    return s[best[1]:best[2]+1] if best[0] != float('inf') else ""

# Slow motion on s="ADOBECODEBANC" t="ABC":
# need={A:1,B:1,C:1}, required=3
# expand to idx=5 (C found): matched=3, window=ADOBEC len=6
# shrink: left=1 (remove A: matched=2), stop
# eventually: best window = BANC (idx 9..12)

def test_harness(fn):
    tests = [
        ("ADOBECODEBANC", "ABC", "BANC"),
        ("a",            "a",  "a"),
        ("a",            "aa", ""),
        ("aa",           "aa", "aa"),
        ("ADOBEC",       "ABC", "ADOBEC"),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(min_window)
print("min_window defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Longest Repeating Character Replacement — LC 424

---

```
PROBLEM:
  Replace at most k characters. Find length of longest window
  where all chars can become the same character.

TRICK:
  Window is valid when: (window_size - max_freq) <= k
  (replacements needed = chars that are NOT the dominant char)
  Key: window NEVER shrinks. When invalid, slide right instead of shrink.
  This locks in the best window size seen so far.

SLOW MOTION TRACE on s="AABABBA" k=1:
  freq={}  left=0  max_freq=0
  r=0 A: freq={A:1} max_f=1 size=1 replacements=0 <=1 ok
  r=1 A: freq={A:2} max_f=2 size=2 replacements=0 ok
  r=2 B: freq={A:2,B:1} max_f=2 size=3 replacements=1 ok
  r=3 A: freq={A:3,B:1} max_f=3 size=4 replacements=1 ok
  r=4 B: freq={A:3,B:2} max_f=3 size=5 replacements=2 >1 → slide window
    remove s[0]='A': freq={A:2,B:2} left=1 size stays 4
  r=5 B: freq={A:2,B:3} max_f=3 size=5 replacements=2 >1 → slide
    remove s[1]='A': freq={A:1,B:3} left=2
  r=6 A: freq={A:2,B:3} max_f=3 size=5 replacements=2 >1 → slide
    remove s[2]='B': left=3
  final size = 4 = answer

KEY INSIGHT:
  Window never shrinks — when invalid we slide (left++ AND right++).
  This means we never revisit a smaller window, saving O(n) time.

TIME:  O(n) — right pointer never retreats
SPACE: O(1) — freq array of 26 chars
```

In [ ]:
def character_replacement(s: str, k: int) -> int:
    """
    LC 424 — Longest Repeating Character Replacement
    Approach: window never shrinks — slide when invalid, grow when valid.
    Args:
        s (str): uppercase English letters.
        k (int): max replacements allowed.
    Returns:
        int: length of longest window achievable.
    Time:  O(n) — each pointer advances at most n steps
    Space: O(1) — 26-char frequency array
    """
    freq = [0] * 26
    left = 0
    max_freq = 0     # max frequency of any char in current window

    for right in range(len(s)):
        freq[ord(s[right]) - ord('A')] += 1
        max_freq = max(max_freq, freq[ord(s[right]) - ord('A')])

        window_size = right - left + 1
        if window_size - max_freq > k:    # too many replacements needed
            # slide window — don't shrink, just move left one step
            freq[ord(s[left]) - ord('A')] -= 1
            left += 1
        # window size is max(best_seen, current) due to slide-not-shrink

    return right - left + 1              # final window size = best found

# Slow motion on "AABABBA" k=1:
# r=4: window=5, max_freq=3, replacements=2>1 → slide left (don't shrink)
# r=5: window=5, max_freq=3, replacements=2>1 → slide left
# r=6: window=5, max_freq=3, replacements=2>1 → slide left
# final: right=6, left=3 → size=4

def test_harness(fn):
    tests = [
        ("ABAB",    2, 4),
        ("AABABBA", 1, 4),
        ("AAAA",    2, 4),
        ("ABCD",    1, 2),
        ("A",       0, 1),
        ("",        0, 0),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(character_replacement)
print("character_replacement defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Fixed Window Maximum — LC 239

---

```
PROBLEM:
  Return max element in each fixed window of size k.

TRICK:
  Monotonic deque stores indices in decreasing-value order.
  Front = current window max. Back = candidates for future max.
  Three rules:
    1. Evict front if it's outside the window (index < i-k+1)
    2. Evict back if back's value <= new value (can never be max again)
    3. Append new index. When window is full, front = window max.

SLOW MOTION TRACE on nums=[1,3,-1,-3,5,3,6,7] k=3:
  i=0 v=1:  dq=[0]         (waiting)
  i=1 v=3:  evict 0(1<3)→ dq=[1]       (waiting)
  i=2 v=-1: dq=[1,2]       window full, max=nums[1]=3
  i=3 v=-3: dq=[1,2,3]     max=nums[1]=3
  i=4 v=5:  evict 1(out),2,3(all<5)→ dq=[4] max=5
  i=5 v=3:  dq=[4,5]       max=nums[4]=5
  i=6 v=6:  evict 4(out),5(3<6)→ dq=[6] max=6
  i=7 v=7:  evict 6(7<7)→ dq=[7]  max=7
  result=[3,3,5,5,6,7]

KEY INSIGHT:
  Once a smaller element is behind a larger one in the window, it can
  NEVER be the max — safe to evict immediately. Hence monotonic.

TIME:  O(n) — each index added and removed from deque at most once
SPACE: O(k) — deque holds at most k indices
```

In [ ]:
from collections import deque
from typing import List

def max_sliding_window(nums: List[int], k: int) -> List[int]:
    """
    LC 239 — Sliding Window Maximum
    Approach: monotonic deque stores indices in decreasing-value order.
    Args:
        nums (List[int]): integer array.
        k (int): window size.
    Returns:
        List[int]: max of each window of size k.
    Time:  O(n) — each index enters and leaves deque at most once
    Space: O(k) — deque holds at most k indices
    """
    dq = deque()    # stores indices; front = current window max
    result = []

    for i, v in enumerate(nums):
        # evict front if it has slid outside the window
        while dq and dq[0] < i - k + 1:
            dq.popleft()

        # evict back elements smaller than new — they can never be max again
        while dq and nums[dq[-1]] <= v:
            dq.pop()

        dq.append(i)

        if i >= k - 1:              # window is full, record maximum
            result.append(nums[dq[0]])

    return result

# Slow motion on [1,3,-1,-3,5,3,6,7] k=3:
# i=0(1):  dq=[0]
# i=1(3):  evict 0 (1<=3) → dq=[1]
# i=2(-1): dq=[1,2] window full → result=[nums[1]]=[ 3]
# i=3(-3): dq=[1,2,3] → result=[3,3]
# i=4(5):  evict 1(out),2,3(all<=5) → dq=[4] → result=[3,3,5]
# ...

def test_harness(fn):
    tests = [
        ([1,3,-1,-3,5,3,6,7], 3, [3,3,5,5,6,7]),
        ([1],                 1, [1]),
        ([1,-1],              1, [1,-1]),
        ([9,11],              2, [11]),
        ([4,-2],              2, [4]),
        ([2,1,5,3,6,4,8,5],  3, [5,5,6,6,8,8]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(max_sliding_window)
print("max_sliding_window defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Permutation in String — LC 567

---

```
PROBLEM:
  Does s2 contain any permutation of s1 as a substring?

TRICK:
  Fixed window of size len(s1). Two 26-bucket frequency arrays.
  Track `matches` = number of positions where s2_freq[i] == s1_freq[i].
  When matches == 26, window is a permutation.

SLOW MOTION TRACE on s1="ab" s2="eidbaooo":
  s1_freq: [1,1,0,...] (a=1,b=1)
  window size = 2

  window [e,i]: s2_freq doesn't match s1_freq anywhere → matches ≠ 26
  window [i,d]: same
  window [d,b]: b matches (s2[b]=1=s1[b]=1) but d doesn't → matches=1
  window [b,a]: a matches AND b matches → matches=26 → return True

KEY INSIGHT:
  Counting matches across 26 buckets avoids comparing full arrays.
  Only update `matches` for the bucket being added/removed.

TIME:  O(|s1| + |s2|) — build + slide
SPACE: O(1) — two 26-element arrays
```

In [ ]:
def check_inclusion(s1: str, s2: str) -> bool:
    """
    LC 567 — Permutation in String
    Approach: fixed window = len(s1), 26-bucket arrays + matches counter.
    Args:
        s1 (str): pattern string.
        s2 (str): source string to search in.
    Returns:
        bool: True if s2 contains any permutation of s1.
    Time:  O(|s1| + |s2|) — build s1 freq + single pass through s2
    Space: O(1) — two fixed-size 26-element arrays
    """
    if len(s1) > len(s2):
        return False

    s1_freq = [0] * 26
    s2_freq = [0] * 26

    for ch in s1:
        s1_freq[ord(ch) - ord('a')] += 1

    # build first window
    for i in range(len(s1)):
        s2_freq[ord(s2[i]) - ord('a')] += 1

    # count matching buckets
    matches = sum(1 for i in range(26) if s1_freq[i] == s2_freq[i])

    if matches == 26:
        return True

    k = len(s1)
    for right in range(k, len(s2)):
        # add right character
        r_idx = ord(s2[right]) - ord('a')
        if s2_freq[r_idx] == s1_freq[r_idx]:    # was matching
            matches -= 1
        s2_freq[r_idx] += 1
        if s2_freq[r_idx] == s1_freq[r_idx]:    # now matching
            matches += 1

        # remove left character
        l_idx = ord(s2[right - k]) - ord('a')
        if s2_freq[l_idx] == s1_freq[l_idx]:    # was matching
            matches -= 1
        s2_freq[l_idx] -= 1
        if s2_freq[l_idx] == s1_freq[l_idx]:    # now matching
            matches += 1

        if matches == 26:
            return True

    return False

# Slow motion on s1="ab" s2="eidbaooo":
# first window [e,i]: matches = 24 (only a and b buckets differ)
# slide to [d,b]: b bucket now matches, matches=25
# slide to [b,a]: a and b both match → matches=26 → True

def test_harness(fn):
    tests = [
        ("ab",  "eidbaooo", True),
        ("ab",  "eidboaoo", False),
        ("adc", "dcda",     True),
        ("a",   "a",        True),
        ("ab",  "a",        False),
        ("ba",  "bbbba",    False),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(check_inclusion)
print("check_inclusion defined.")

<a id='10'></a>

## 10. The Sliding Window Decision Map

```
QUESTION TYPE                           KEY TECHNIQUE            LC PROBLEMS
──────────────────────────────────────────────────────────────────────────────
Longest substring without repeats       Variable + last_seen map  3
Min window containing target chars      Variable + two freq maps  76
Longest window after k replacements     Slide-not-shrink trick    424
Max/min in each window of size k        Fixed + monotonic deque   239
Does s2 contain permutation of s1       Fixed + 26-bucket match   567
Longest substring with at most k dist  Variable + dict cleanup   340, 159

TEMPLATE CHOICE:
  need maximum of window of EXACT size k  → fixed window
  need longest/shortest satisfying cond   → variable window
  cannot let window shrink                → slide-not-shrink (LC 424)
```

<a id='11'></a>

## 11. Interview Cheat Sheet

**1. When to reach for Sliding Window:**

| Signal | What To Do |
|--------|------------|
| Longest/shortest subarray or substring | Variable window |
| All windows of fixed size k | Fixed window |
| Max/min of each window | Monotonic deque |
| Replace k chars, maximize length | Slide-not-shrink |
| Permutation / anagram check | Fixed + freq arrays |

**2. Core operations — memorize these:**

```python
left = 0
for right in range(len(arr)):
    # add arr[right] to window
    while window_invalid():
        # remove arr[left] from window
        left += 1
    best = max(best, right - left + 1)
```

**3. Common templates:**

```python
# TEMPLATE 1: VARIABLE WINDOW (longest valid)
left = 0; best = 0; window = defaultdict(int)
for right, ch in enumerate(s):
    window[ch] += 1
    while len(window) > k:  # shrink when invalid
        window[s[left]] -= 1
        if window[s[left]] == 0: del window[s[left]]
        left += 1
    best = max(best, right - left + 1)

# TEMPLATE 2: FIXED WINDOW MAX (monotonic deque)
dq = deque()
for i, v in enumerate(nums):
    while dq and dq[0] < i - k + 1: dq.popleft()
    while dq and nums[dq[-1]] <= v: dq.pop()
    dq.append(i)
    if i >= k - 1: result.append(nums[dq[0]])

# TEMPLATE 3: SLIDE NOT SHRINK (LC 424)
left = 0; max_freq = 0; freq = [0]*26
for right in range(len(s)):
    freq[ord(s[right])-65] += 1
    max_freq = max(max_freq, freq[ord(s[right])-65])
    if right - left + 1 - max_freq > k:  # invalid: slide
        freq[ord(s[left])-65] -= 1; left += 1
```

**4. Gotchas:**

```
❌  Shrinking window in LC 424 — slide instead
❌  Forgetting to remove zero-count entries from freq map
❌  Off-by-one in fixed window: window full when i >= k-1
❌  Evicting deque front by value not index — use index check
✅  Variable window: shrink WHILE invalid, not IF
✅  Monotonic deque: check evict before append
```

<a id='12'></a>

## 12. Summary Map

```
SLIDING WINDOW
│
├── Variable Window (constraint-driven)
│     ├── Set / HashMap tracks window contents
│     ├── Expand right always, shrink left on violation
│     └── LC 3, LC 76, LC 340
│
├── Slide-Not-Shrink (maximize window size)
│     ├── Window only moves forward — never shrinks
│     └── LC 424
│
└── Fixed Window
      ├── Size is always k — evict when sliding past
      ├── With monotonic deque → O(1) max per window
      └── LC 239, LC 567

CORE INVARIANT:
  The window [left, right] always satisfies the constraint.
  Right moves forward; left moves forward only when needed.
  Each element enters and exits the window exactly once → O(n).
```

---
*End of Sliding Window Master Guide — Sean Edition*